# Task 1 — Kaggle launcher

Thin launcher only: all logic is in `task1/` and `kaggle/task1_queue.py`.

Settings: **GPU T4 x2**, **Internet on**, secret **HF_TOKEN** (Add-ons → Secrets). After any interruption just run all cells again: every job resumes from its newest synced checkpoint.

In [ ]:
REPO_URL= "https://github.com/Zeeba1705/atml_PA3.git"
BRANCH= "task1-finetune"
SYNC_REPO= "zeeba1705/atml-pa3"   #private HF Hub repo, created on first use
#12 h session limit minus setup time
MAX_HOURS= 11.0

In [ ]:
import os
import subprocess
import sys
import time

if not os.path.exists("/kaggle/working/atml_PA3"):
    !git clone -b {BRANCH} {REPO_URL} /kaggle/working/atml_PA3

%cd /kaggle/working/atml_PA3
!git fetch -q && git checkout -q {BRANCH} && git pull -q
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch, transformers, numpy; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'transformers', transformers.__version__, 'numpy', numpy.__version__)"

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"]= UserSecretsClient().get_secret("HF_TOKEN")

### Data prep (CPU, ~1 min)
Rebuilds `train_tokenized.pt`; `task1.train` refuses to run if it doesn't match `splits/task1/train_selection.json`.

In [ ]:
!python -m task1.data

### Status

In [ ]:
!python kaggle/task1_queue.py --status --sync_repo {SYNC_REPO}

### Run both queues (one per T4)
`gpu0`: full_ft → l2sp → 4 evals, `gpu1`: lora16 → lora4 → oplora16 → 3 evals. Logs in `logs/`.

In [ ]:
os.makedirs("logs", exist_ok=True)

#print training lines as they happen (stdout to a file is buffered) and drop the HF upload progress bars
os.environ["PYTHONUNBUFFERED"]= "1"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"]= "1"

procs= {}
for group, gpu in [("gpu0", 0), ("gpu1", 1)]:
    log= open(f"logs/{group}.log", "a")
    procs[group]= subprocess.Popen(
        [
            sys.executable, "kaggle/task1_queue.py",
            "--group", group,
            "--gpu", str(gpu),
            "--sync_repo", SYNC_REPO,
            "--max_hours", str(MAX_HOURS)
        ],
        stdout=log,
        stderr=subprocess.STDOUT
    )

#keep the notebook alive (needed for Save & Run All) and copy every new log line here every 2 min
seen= {group: 0 for group in procs}
while True:
    running= any(p.poll() is None for p in procs.values())
    time.sleep(120 if running else 0)

    for group in procs:
        with open(f"logs/{group}.log") as f:
            f.seek(seen[group])
            new= f.read()
            seen[group]= f.tell()

        for line in new.splitlines():
            print(f"[{group}] {line}")

    if not running:
        break

print({group: p.returncode for group, p in procs.items()})

In [ ]:
!python kaggle/task1_queue.py --status --sync_repo {SYNC_REPO}